In [1]:
#Import libds
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("data/water_data_full_combined.csv", low_memory=False)

In [3]:
# Lọc hồ
lake = df[df["type"].str.lower() == "lake"].copy()

# Chuyển thời gian
lake["timestamp"] = pd.to_datetime(lake["Thời gian (UTC)"], errors="coerce")

# Sắp xếp
lake = lake.sort_values(["Mã trạm/LakeCode", "timestamp"]).reset_index(drop=True)

# Đổi tên cột cho dễ xử lý
lake = lake.rename(columns={
    "Mã trạm/LakeCode": "lake_code",
    "Trạm/Hồ": "lake_name",
    "Tên sông/Lưu vực": "basin_name",
    "Tên tỉnh": "province",
    "Mực nước (m)": "water_level",
    "Dung tích (m3)": "storage",
    "Dung tích TK (m3)": "designed_storage",
    "Tỷ lệ dung tích (%)": "storage_pct",
    "Q đến (m3/s)": "inflow",
    "Q xả (m3/s)": "outflow",
    "Mực nước BT (m)": "normal_level",
    "Mực nước GC (m)": "dead_level",
    "Cảnh báo/Xu thế": "trend_text"
})

num_cols = [
    "water_level", "storage", "designed_storage", "storage_pct",
    "inflow", "outflow", "normal_level", "dead_level",
    "x", "y", "province_code", "basin_code"
]

for c in num_cols:
    if c in lake.columns:
        lake[c] = pd.to_numeric(lake[c], errors="coerce")

print(lake.shape)
print(lake[["lake_code","lake_name", "timestamp","province", "Thời gian (UTC)","water_level"]].head())

(58575, 28)
                              lake_code   lake_name           timestamp  \
0  062A7CF0-46F3-4E99-8BCD-040CEF304344  Thuận Ninh 2014-08-06 07:00:00   
1  062A7CF0-46F3-4E99-8BCD-040CEF304344  Thuận Ninh 2014-08-07 07:00:00   
2  062A7CF0-46F3-4E99-8BCD-040CEF304344  Thuận Ninh 2014-08-08 07:00:00   
3  062A7CF0-46F3-4E99-8BCD-040CEF304344  Thuận Ninh 2014-08-11 07:00:00   
4  062A7CF0-46F3-4E99-8BCD-040CEF304344  Thuận Ninh 2014-08-12 07:00:00   

  province   Thời gian (UTC)  water_level  
0  Gia Lai  2014-08-06 07:00        61.57  
1  Gia Lai  2014-08-07 07:00        61.53  
2  Gia Lai  2014-08-08 07:00        61.48  
3  Gia Lai  2014-08-11 07:00        61.35  
4  Gia Lai  2014-08-12 07:00        61.29  


In [4]:
# Data Quanlity Check
missing_ratio = lake.isna().mean().sort_values(ascending=False)
print(missing_ratio)

missing_by_lake = lake.groupby("lake_name").apply(lambda x: x.isna().mean())
print(missing_by_lake[["water_level", "inflow", "outflow", "storage_pct", "normal_level", "dead_level"]])

Năm lũ lịch sử             1.000000
Cảnh báo value (0-4)       1.000000
Mực nước lịch sử (m)       1.000000
BĐ3 (m)                    1.000000
BĐ2 (m)                    1.000000
BĐ1 (m)                    1.000000
Chênh lệch cảnh báo (m)    1.000000
dead_level                 0.447580
outflow                    0.354008
Mã Cảnh báo                0.241093
trend_text                 0.240973
normal_level               0.239932
inflow                     0.145181
x                          0.034127
y                          0.034127
water_level                0.000034
storage_pct                0.000000
province_code              0.000000
basin_code                 0.000000
type                       0.000000
designed_storage           0.000000
storage                    0.000000
lake_code                  0.000000
Thời gian (UTC)            0.000000
province                   0.000000
basin_name                 0.000000
lake_name                  0.000000
timestamp                  0

In [5]:
print("Duplicate full rows:", lake.duplicated().sum())
print("Duplicate lake_code + timestamp:", lake.duplicated(subset=["lake_code", "timestamp"]).sum())

Duplicate full rows: 0
Duplicate lake_code + timestamp: 0


In [6]:
lake["time_diff_hours"] = (
    lake.groupby("lake_code")["timestamp"]
        .diff()
        .dt.total_seconds() / 3600
)

print(lake["time_diff_hours"].describe())
print(lake["time_diff_hours"].value_counts().head(10))

count    58556.000000
mean        27.556038
std         84.524767
min          0.166667
25%         24.000000
50%         24.000000
75%         24.000000
max       8492.000000
Name: time_diff_hours, dtype: float64
time_diff_hours
24.000000    36072
22.000000     2962
26.000000     2932
20.000000     1280
28.000000     1180
18.000000      795
30.000000      780
23.000000      765
25.000000      703
24.166667      687
Name: count, dtype: int64


In [7]:
# Chuẩn hoá dữ liệu thời gian 
lake_daily = (
    lake.set_index("timestamp")
        .groupby("lake_code")
        .resample("D")
        .agg({
            "lake_name": "last",
            "province": "last",
            "basin_name": "last",
            "water_level": "mean",
            "inflow": "mean",
            "outflow": "mean",
            "storage": "last",
            "designed_storage": "last",
            "storage_pct": "last",
            "normal_level": "last",
            "dead_level": "last",
            "trend_text": "last",
            "x": "last",
            "y": "last",
            "province_code": "last",
            "basin_code": "last"
        })
        .reset_index()
)

print(lake_daily.head())
print(lake_daily.shape)

                              lake_code  timestamp   lake_name province  \
0  062A7CF0-46F3-4E99-8BCD-040CEF304344 2014-08-06  Thuận Ninh  Gia Lai   
1  062A7CF0-46F3-4E99-8BCD-040CEF304344 2014-08-07  Thuận Ninh  Gia Lai   
2  062A7CF0-46F3-4E99-8BCD-040CEF304344 2014-08-08  Thuận Ninh  Gia Lai   
3  062A7CF0-46F3-4E99-8BCD-040CEF304344 2014-08-09         NaN      NaN   
4  062A7CF0-46F3-4E99-8BCD-040CEF304344 2014-08-10         NaN      NaN   

       basin_name  water_level  inflow  outflow    storage  designed_storage  \
0  Kôn - Hà Thanh        61.57     NaN      NaN  18.074020             35.36   
1  Kôn - Hà Thanh        61.53     NaN      NaN  17.966492             35.36   
2  Kôn - Hà Thanh        61.48     NaN      NaN  17.832077             35.36   
3             NaN          NaN     NaN      NaN        NaN               NaN   
4             NaN          NaN     NaN      NaN        NaN               NaN   

   storage_pct  normal_level  dead_level trend_text           x     

In [8]:
print(lake_daily.columns.tolist())


['lake_code', 'timestamp', 'lake_name', 'province', 'basin_name', 'water_level', 'inflow', 'outflow', 'storage', 'designed_storage', 'storage_pct', 'normal_level', 'dead_level', 'trend_text', 'x', 'y', 'province_code', 'basin_code']


In [9]:
def fill_time_series(g):
    g = g.sort_values("timestamp").copy()

    static_text_cols = ["lake_name", "province", "basin_name"]
    static_num_cols = [
        "designed_storage", "normal_level", "dead_level",
        "x", "y", "province_code", "basin_code"
    ]

    for col in static_text_cols + static_num_cols:
        if col in g.columns:
            g[col] = g[col].ffill().bfill()

    for col in ["water_level", "storage", "storage_pct"]:
        if col in g.columns:
            g[col] = g[col].interpolate(
                method="linear",
                limit=3,
                limit_direction="both",
                limit_area="inside"
            )

    for col in ["inflow", "outflow"]:
        if col in g.columns:
            g[f"{col}_missing"] = g[col].isna().astype(int)
            g[col] = g[col].interpolate(
                method="linear",
                limit=2,
                limit_direction="both",
                limit_area="inside"
            )

    if {"storage", "designed_storage"}.issubset(g.columns):
        ratio = (g["storage"] / g["designed_storage"] * 100)
        g["storage_pct"] = g["storage_pct"].fillna(ratio)

    return g

lake_daily = (
    lake_daily.groupby("lake_code", group_keys=False)
              .apply(lambda g: fill_time_series(g).assign(lake_code=g.name))
              .reset_index(drop=True)
)
print(lake_daily.columns.tolist())
print(lake_daily.head())

['timestamp', 'lake_name', 'province', 'basin_name', 'water_level', 'inflow', 'outflow', 'storage', 'designed_storage', 'storage_pct', 'normal_level', 'dead_level', 'trend_text', 'x', 'y', 'province_code', 'basin_code', 'inflow_missing', 'outflow_missing', 'lake_code']
   timestamp   lake_name province      basin_name  water_level  inflow  \
0 2014-08-06  Thuận Ninh  Gia Lai  Kôn - Hà Thanh    61.570000     NaN   
1 2014-08-07  Thuận Ninh  Gia Lai  Kôn - Hà Thanh    61.530000     NaN   
2 2014-08-08  Thuận Ninh  Gia Lai  Kôn - Hà Thanh    61.480000     NaN   
3 2014-08-09  Thuận Ninh  Gia Lai  Kôn - Hà Thanh    61.436667     NaN   
4 2014-08-10  Thuận Ninh  Gia Lai  Kôn - Hà Thanh    61.393333     NaN   

   outflow    storage  designed_storage  storage_pct  normal_level  \
0      NaN  18.074020             35.36    51.114310          68.0   
1      NaN  17.966492             35.36    50.810215          68.0   
2      NaN  17.832077             35.36    50.430084          68.0   
3    

In [10]:
suspect = lake_daily[lake_daily["water_level"] > 1200][
    ["lake_name", "timestamp", "water_level"]
].sort_values("water_level", ascending=False)

print(suspect)

Empty DataFrame
Columns: [lake_name, timestamp, water_level]
Index: []


In [11]:
g = lake_daily[lake_daily["lake_name"] == "Ayun Hạ"].sort_values("timestamp")
target_time = pd.Timestamp("2022-02-11 19:00:00")

print(
    g[(g["timestamp"] >= target_time - pd.Timedelta(days=3)) &
      (g["timestamp"] <= target_time + pd.Timedelta(days=3))]
    [["timestamp", "water_level", "inflow", "outflow", "storage_pct"]]
)

       timestamp  water_level  inflow  outflow  storage_pct
42950 2022-02-09       202.90    2.63      0.0    90.955450
42951 2022-02-10       202.84    1.45      0.0    90.462160
42952 2022-02-11       202.80    1.46      0.0   107.030000
42953 2022-02-12       202.78    3.86      0.0    89.968864
42954 2022-02-13       202.72    3.84      0.0    89.475586
42955 2022-02-14       202.68    2.65      0.0    89.146600


In [12]:
#EDA dữ liệu 
summary_by_lake = lake_daily.groupby("lake_name").agg(
    n_days=("timestamp", "size"),
    start=("timestamp", "min"),
    end=("timestamp", "max"),
    water_missing=("water_level", lambda s: s.isna().mean()),
    inflow_missing=("inflow", lambda s: s.isna().mean()),
    outflow_missing=("outflow", lambda s: s.isna().mean())
).sort_values("n_days", ascending=False)

print(summary_by_lake)


                n_days      start        end  water_missing  inflow_missing  \
lake_name                                                                     
Định Bình         4745 2013-01-31 2026-01-27       0.013488        0.088303   
Ia MLá            4420 2013-12-22 2026-01-27       0.053846        0.171267   
Thuận Ninh        4193 2014-08-06 2026-01-27       0.074171        0.493441   
Tả Trạch          4188 2014-08-11 2026-01-27       0.076409        0.218959   
Pleikrông         4176 2014-08-23 2026-01-27       0.083573        0.131466   
Sông Hinh         4176 2014-08-23 2026-01-27       0.194923        0.207375   
A Vương           4176 2014-08-23 2026-01-27       0.072318        0.134339   
Kanak             4176 2014-08-23 2026-01-27       0.002155        0.061782   
Ialy              4176 2014-08-23 2026-01-27       0.084770        0.132423   
SeSan4            4176 2014-08-23 2026-01-27       0.025623        0.131466   
Sông Tranh 2      4162 2014-08-23 2026-01-13       0

In [13]:
print(lake_daily.columns.tolist())

['timestamp', 'lake_name', 'province', 'basin_name', 'water_level', 'inflow', 'outflow', 'storage', 'designed_storage', 'storage_pct', 'normal_level', 'dead_level', 'trend_text', 'x', 'y', 'province_code', 'basin_code', 'inflow_missing', 'outflow_missing', 'lake_code']


In [14]:
summary_by_lake = lake_daily.groupby("lake_name").agg(
    n_days=("timestamp", "size"),
    start=("timestamp", "min"),
    end=("timestamp", "max"),
    water_missing=("water_level", lambda s: s.isna().mean()),
    inflow_missing=("inflow", lambda s: s.isna().mean()),
    outflow_missing=("outflow", lambda s: s.isna().mean())
).sort_values("n_days", ascending=False)

print(summary_by_lake)

                n_days      start        end  water_missing  inflow_missing  \
lake_name                                                                     
Định Bình         4745 2013-01-31 2026-01-27       0.013488        0.088303   
Ia MLá            4420 2013-12-22 2026-01-27       0.053846        0.171267   
Thuận Ninh        4193 2014-08-06 2026-01-27       0.074171        0.493441   
Tả Trạch          4188 2014-08-11 2026-01-27       0.076409        0.218959   
Pleikrông         4176 2014-08-23 2026-01-27       0.083573        0.131466   
Sông Hinh         4176 2014-08-23 2026-01-27       0.194923        0.207375   
A Vương           4176 2014-08-23 2026-01-27       0.072318        0.134339   
Kanak             4176 2014-08-23 2026-01-27       0.002155        0.061782   
Ialy              4176 2014-08-23 2026-01-27       0.084770        0.132423   
SeSan4            4176 2014-08-23 2026-01-27       0.025623        0.131466   
Sông Tranh 2      4162 2014-08-23 2026-01-13       0

In [15]:
lake_daily["water_level_z"] = (
    lake_daily.groupby("lake_code")["water_level"]
              .transform(lambda s: (s - s.mean()) / s.std() if s.std() else 0)
)

In [16]:
lake_daily["month"] = lake_daily["timestamp"].dt.month
lake_daily["year"] = lake_daily["timestamp"].dt.year
lake_daily["dayofyear"] = lake_daily["timestamp"].dt.dayofyear

monthly_pattern = lake_daily.groupby("month").agg(
    mean_level_z=("water_level_z", "mean"),
    mean_inflow=("inflow", "mean"),
    mean_outflow=("outflow", "mean")
)
print(monthly_pattern)

       mean_level_z  mean_inflow  mean_outflow
month                                         
1          0.804087    94.904322     78.745286
2          0.676215    58.408829     67.025825
3          0.402277    55.932070     59.860372
4          0.081601    52.321775     56.850020
5         -0.166554    58.880235     68.564320
6         -0.344853    77.137745     92.182287
7         -0.632376    93.161660     85.953723
8         -0.756149   110.523743    114.981444
9         -0.743386   133.823698    131.557873
10        -0.286337   170.333139    184.932395
11         0.291040   184.096880    174.060472
12         0.708438   132.924310    104.962152


In [17]:
# xu thế biên động 
lake_daily["delta_level_1d"] = lake_daily.groupby("lake_code")["water_level"].diff(1)
lake_daily["delta_level_3d"] = lake_daily.groupby("lake_code")["water_level"].diff(3)

lake_daily["rolling_mean_3"] = (
    lake_daily.groupby("lake_code")["water_level"]
              .transform(lambda s: s.rolling(3).mean())
)

lake_daily["rolling_std_7"] = (
    lake_daily.groupby("lake_code")["water_level"]
              .transform(lambda s: s.rolling(7).std())
)

In [18]:
corrs = []
for lake_name, g in lake_daily.groupby("lake_name"):
    sub = g[["water_level", "inflow", "outflow", "storage_pct"]].dropna()
    if len(sub) > 20:
        c = sub.corr()["water_level"]
        corrs.append({
            "lake_name": lake_name,
            "corr_inflow": c.get("inflow", np.nan),
            "corr_outflow": c.get("outflow", np.nan),
            "corr_storage_pct": c.get("storage_pct", np.nan),
        })

corrs = pd.DataFrame(corrs)
print(corrs.sort_values("corr_inflow", ascending=False))

         lake_name  corr_inflow  corr_outflow  corr_storage_pct
1          Ayun Hạ     0.394030      0.288915          0.912183
8           SeSan4     0.356493      0.426396          1.000000
12    Sông Tranh 2     0.240307      0.268942          1.000000
16        Tả Trạch     0.204106      0.275754          0.955281
0          A Vương     0.186266      0.212198          0.999887
5            Kanak     0.170688      0.095397          0.975658
4             Ialy     0.166257      0.177373          1.000000
15         Trà Xom     0.153586      0.077796          0.349507
3           Ia MLá     0.149836      0.202095          0.989118
2       Hương Điền     0.138928      0.370533          0.999530
10     Sông Bung 4     0.121121      0.233436          1.000000
9       Sông Ba Hạ     0.102427      0.079783          0.995835
14  Thượng Kon Tum     0.077075     -0.055944          0.998651
17       Định Bình     0.066105     -0.053373          0.986778
6       Nước Trong     0.059221      0.0

In [19]:
# Data mining
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPORT_DIR = Path("reports/eda_lake")
(REPORT_DIR / "01_data_overview").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "02_missing").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "03_time_coverage").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "04_distribution").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "05_seasonality").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "06_trend").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "07_correlation").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "08_anomaly").mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "09_data_mining").mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

def save_fig(path, dpi=200):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()

def top_n_value_counts(df, col, n=20):
    return df[col].value_counts().head(n)

In [20]:
lake_counts = lake_daily["lake_name"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(14, 7))
lake_counts.head(20).plot(kind="bar")
plt.title("Top 20 hồ có nhiều bản ghi nhất")
plt.xlabel("Hồ")
plt.ylabel("Số bản ghi")
plt.xticks(rotation=45, ha="right")
save_fig(REPORT_DIR / "01_data_overview" / "top20_lakes_by_records.png")

In [21]:
province_counts = lake_daily["province"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(14, 7))
province_counts.head(20).plot(kind="bar")
plt.title("Top 20 tỉnh có nhiều bản ghi hồ nhất")
plt.xlabel("Tỉnh")
plt.ylabel("Số bản ghi")
plt.xticks(rotation=45, ha="right")
save_fig(REPORT_DIR / "01_data_overview" / "top20_provinces_by_records.png")

In [22]:
# Missing value
missing_ratio = lake_daily.isna().mean().sort_values(ascending=False)

plt.figure(figsize=(14, 7))
missing_ratio.plot(kind="bar")
plt.title("Tỷ lệ thiếu dữ liệu theo cột")
plt.xlabel("Biến")
plt.ylabel("Tỷ lệ thiếu")
plt.xticks(rotation=45, ha="right")
save_fig(REPORT_DIR / "02_missing" / "missing_ratio_by_column.png")

In [23]:
important_cols = ["water_level", "inflow", "outflow", "storage_pct", "normal_level", "dead_level"]

missing_by_lake = (
    lake_daily.groupby("lake_name")[important_cols]
    .apply(lambda x: x.isna().mean())
)

missing_by_lake.to_csv(REPORT_DIR / "02_missing" / "missing_by_lake.csv", encoding="utf-8-sig")

In [24]:
sample_lakes = lake_daily["lake_name"].value_counts().head(10).index.tolist()
heat_df = lake_daily[lake_daily["lake_name"].isin(sample_lakes)].copy()

heat_data = heat_df[important_cols].isna().astype(int).values

plt.figure(figsize=(14, 8))
plt.imshow(heat_data[:1000], aspect="auto", interpolation="nearest")
plt.title("Heatmap missing values (10 hồ phổ biến, 1000 dòng đầu)")
plt.xlabel("Biến")
plt.ylabel("Dòng dữ liệu")
plt.xticks(range(len(important_cols)), important_cols, rotation=45, ha="right")
plt.colorbar(label="Missing (1 = thiếu)")
save_fig(REPORT_DIR / "02_missing" / "missing_heatmap_sample.png")

In [25]:
coverage = lake_daily.groupby("lake_name").agg(
    start_date=("timestamp", "min"),
    end_date=("timestamp", "max"),
    n_days=("timestamp", "size")
).sort_values("start_date")

coverage.to_csv(REPORT_DIR / "03_time_coverage" / "lake_time_coverage.csv", encoding="utf-8-sig")

In [26]:
plt.figure(figsize=(14, 10))
y_pos = np.arange(len(coverage))
plt.hlines(y=y_pos, xmin=coverage["start_date"], xmax=coverage["end_date"])
plt.yticks(y_pos, coverage.index)
plt.title("Khoảng thời gian dữ liệu theo từng hồ")
plt.xlabel("Thời gian")
plt.ylabel("Hồ")
save_fig(REPORT_DIR / "03_time_coverage" / "lake_time_coverage.png")

In [27]:
lake_daily = lake_daily.sort_values(["lake_code", "timestamp"]).copy()
lake_daily["time_diff_hours"] = (
    lake_daily.groupby("lake_code")["timestamp"]
    .diff()
    .dt.total_seconds() / 3600
)

plt.figure(figsize=(12, 6))
lake_daily["time_diff_hours"].dropna().clip(upper=72).hist(bins=50)
plt.title("Phân phối khoảng cách thời gian giữa 2 bản ghi liên tiếp")
plt.xlabel("Số giờ")
plt.ylabel("Tần suất")
save_fig(REPORT_DIR / "03_time_coverage" / "time_diff_hours_distribution.png")

In [28]:
num_features = ["water_level", "inflow", "outflow", "storage_pct"]

for col in num_features:
    plt.figure(figsize=(12, 6))
    lake_daily[col].dropna().hist(bins=50)
    plt.title(f"Histogram - {col}")
    plt.xlabel(col)
    plt.ylabel("Tần suất")
    save_fig(REPORT_DIR / "04_distribution" / f"hist_{col}.png")

In [29]:
for col in num_features:
    plt.figure(figsize=(10, 5))
    plt.boxplot(lake_daily[col].dropna(), vert=False)
    plt.title(f"Boxplot - {col}")
    plt.xlabel(col)
    save_fig(REPORT_DIR / "04_distribution" / f"boxplot_{col}.png")

In [30]:
top_lakes = lake_daily["lake_name"].value_counts().head(6).index.tolist()

for lake_name in top_lakes:
    g = lake_daily[lake_daily["lake_name"] == lake_name]
    plt.figure(figsize=(12, 6))
    g["water_level"].dropna().hist(bins=40)
    plt.title(f"Histogram mực nước - {lake_name}")
    plt.xlabel("water_level")
    plt.ylabel("Tần suất")
    safe_name = str(lake_name).replace("/", "_").replace(" ", "_")
    save_fig(REPORT_DIR / "04_distribution" / f"hist_water_level_{safe_name}.png")

In [31]:
lake_daily["month"] = lake_daily["timestamp"].dt.month

monthly_stats = lake_daily.groupby("month").agg(
    mean_water_level=("water_level", "mean"),
    mean_inflow=("inflow", "mean"),
    mean_outflow=("outflow", "mean"),
    mean_storage_pct=("storage_pct", "mean")
)

monthly_stats.to_csv(REPORT_DIR / "05_seasonality" / "monthly_stats.csv", encoding="utf-8-sig")

In [32]:
for col in monthly_stats.columns:
    plt.figure(figsize=(12, 6))
    monthly_stats[col].plot(marker="o")
    plt.title(f"Biến động theo tháng - {col}")
    plt.xlabel("Tháng")
    plt.ylabel(col)
    save_fig(REPORT_DIR / "05_seasonality" / f"monthly_{col}.png")

In [33]:
lake_daily["year"] = lake_daily["timestamp"].dt.year

yearly_level = lake_daily.groupby("year")["water_level"].mean()

plt.figure(figsize=(12, 6))
yearly_level.plot(marker="o")
plt.title("Mực nước trung bình theo năm")
plt.xlabel("Năm")
plt.ylabel("Mực nước trung bình")
save_fig(REPORT_DIR / "05_seasonality" / "yearly_mean_water_level.png")

In [34]:
for lake_name in top_lakes:
    g = lake_daily[lake_daily["lake_name"] == lake_name].sort_values("timestamp").copy()
    g["rolling_mean_7"] = g["water_level"].rolling(7).mean()

    plt.figure(figsize=(14, 6))
    plt.plot(g["timestamp"], g["water_level"], label="water_level")
    plt.plot(g["timestamp"], g["rolling_mean_7"], label="rolling_mean_7")
    plt.title(f"Chuỗi thời gian mực nước - {lake_name}")
    plt.xlabel("Thời gian")
    plt.ylabel("Mực nước")
    plt.legend()
    safe_name = str(lake_name).replace("/", "_").replace(" ", "_")
    save_fig(REPORT_DIR / "06_trend" / f"trend_water_level_{safe_name}.png")

In [35]:
lake_daily["delta_level_1d"] = lake_daily.groupby("lake_code")["water_level"].diff(1)

plt.figure(figsize=(12, 6))
lake_daily["delta_level_1d"].dropna().clip(-5, 5).hist(bins=60)
plt.title("Phân phối biến động mực nước theo ngày")
plt.xlabel("delta_level_1d")
plt.ylabel("Tần suất")
save_fig(REPORT_DIR / "06_trend" / "hist_delta_level_1d.png")

In [36]:
corr_cols = ["water_level", "inflow", "outflow", "storage_pct", "normal_level", "dead_level", "delta_level_1d"]
corr_df = lake_daily[corr_cols].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr_df, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title("Correlation heatmap - toàn bộ dữ liệu hồ")
save_fig(REPORT_DIR / "07_correlation" / "global_correlation_heatmap.png")

corr_df.to_csv(REPORT_DIR / "07_correlation" / "global_correlation_matrix.csv", encoding="utf-8-sig")

In [37]:
for lake_name in top_lakes:
    g = lake_daily[lake_daily["lake_name"] == lake_name][corr_cols].dropna()
    if len(g) < 20:
        continue

    corr = g.corr()
    plt.figure(figsize=(8, 6))
    plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
    plt.yticks(range(len(corr_cols)), corr_cols)
    plt.title(f"Correlation heatmap - {lake_name}")
    safe_name = str(lake_name).replace("/", "_").replace(" ", "_")
    save_fig(REPORT_DIR / "07_correlation" / f"corr_{safe_name}.png")

In [38]:
def find_iqr_outliers(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return df[(df[col] < lower) | (df[col] > upper)]

outliers_level = find_iqr_outliers(lake_daily.dropna(subset=["water_level"]), "water_level")
outliers_level.to_csv(REPORT_DIR / "08_anomaly" / "outliers_water_level_iqr.csv", index=False, encoding="utf-8-sig")

In [39]:
plt.figure(figsize=(14, 6))
plt.scatter(lake_daily["timestamp"], lake_daily["water_level"], s=5, alpha=0.4, label="Normal")
plt.scatter(outliers_level["timestamp"], outliers_level["water_level"], s=8, alpha=0.8, label="Outlier")
plt.title("Outlier mực nước theo thời gian")
plt.xlabel("Thời gian")
plt.ylabel("water_level")
plt.legend()
save_fig(REPORT_DIR / "08_anomaly" / "outlier_water_level_scatter.png")

In [40]:
lake_daily["inflow_state"] = pd.qcut(lake_daily["inflow"], q=4, labels=["low", "mid", "high", "very_high"])
lake_daily["water_level_state"] = pd.qcut(lake_daily["water_level"], q=4, labels=["low", "mid", "high", "very_high"])
lake_daily["delta_state"] = pd.qcut(lake_daily["delta_level_1d"].dropna(), q=4, labels=["fall", "stable", "rise", "sharp_rise"])

In [41]:
for col in ["inflow_state", "water_level_state"]:
    plt.figure(figsize=(8, 5))
    lake_daily[col].value_counts().sort_index().plot(kind="bar")
    plt.title(f"Tần suất trạng thái - {col}")
    plt.xlabel("Trạng thái")
    plt.ylabel("Số lượng")
    save_fig(REPORT_DIR / "09_data_mining" / f"state_frequency_{col}.png")

In [42]:
combo = (
    lake_daily[["inflow_state", "water_level_state"]]
    .dropna()
    .value_counts()
    .head(10)
)
print(combo.head(5))

plt.figure(figsize=(10, 6))
combo.plot(kind="bar")
plt.title("Top 10 tổ hợp trạng thái phổ biến")
plt.xlabel("(inflow_state, water_level_state)")
plt.ylabel("Số lượng")
plt.xticks(rotation=45, ha="right")
save_fig(REPORT_DIR / "09_data_mining" / "top10_state_combinations.png")

inflow_state  water_level_state
very_high     very_high            4578
              high                 4543
high          mid                  4410
low           very_high            4178
mid           mid                  3642
Name: count, dtype: int64


In [43]:
# Forecast feature engineering + auto feature selection + baseline + LightGBM/XGBoost
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import mutual_info_regression

REPORT_DIR = Path("reports/eda_lake")
top_lake_counts = lake_daily["lake_name"].value_counts().sort_values(ascending=False)
top_lake_name = top_lake_counts.index[0]
top_lake_n = int(top_lake_counts.iloc[0])
safe_lake_name = str(top_lake_name).replace("/", "_").replace(" ", "_")
FORECAST_DIR = REPORT_DIR / "10_forecast" / safe_lake_name
FORECAST_DIR.mkdir(parents=True, exist_ok=True)
forecast_df = (
    lake_daily[lake_daily["lake_name"] == top_lake_name]
    .sort_values(["lake_code", "timestamp"])
    .copy()
)

forecast_df["dayofyear"] = forecast_df["timestamp"].dt.dayofyear
forecast_df["month"] = forecast_df["timestamp"].dt.month
forecast_df["weekofyear"] = forecast_df["timestamp"].dt.isocalendar().week.astype(int)
forecast_df["dayofweek"] = forecast_df["timestamp"].dt.dayofweek
forecast_df["is_weekend"] = (forecast_df["dayofweek"] >= 5).astype(int)
forecast_df["month_sin"] = np.sin(2 * np.pi * forecast_df["month"] / 12)
forecast_df["month_cos"] = np.cos(2 * np.pi * forecast_df["month"] / 12)
forecast_df["doy_sin"] = np.sin(2 * np.pi * forecast_df["dayofyear"] / 365.25)
forecast_df["doy_cos"] = np.cos(2 * np.pi * forecast_df["dayofyear"] / 365.25)

forecast_df["net_flow"] = forecast_df["inflow"] - forecast_df["outflow"]
forecast_df["level_above_dead"] = forecast_df["water_level"] - forecast_df["dead_level"]
forecast_df["level_to_normal_gap"] = forecast_df["normal_level"] - forecast_df["water_level"]
level_range = (forecast_df["normal_level"] - forecast_df["dead_level"]).replace(0, np.nan)
forecast_df["normalized_level"] = (forecast_df["water_level"] - forecast_df["dead_level"]) / level_range
forecast_df["storage_pct_delta_1d"] = forecast_df.groupby("lake_code")["storage_pct"].diff(1)

for col in ["water_level", "inflow", "outflow", "storage_pct", "net_flow", "normalized_level", "level_above_dead", "level_to_normal_gap"]:
    if col not in forecast_df.columns:
        continue
    for lag in [1, 3, 7, 14, 30]:
        forecast_df[f"{col}_lag_{lag}"] = forecast_df.groupby("lake_code")[col].shift(lag)

for col in ["water_level", "inflow", "outflow", "net_flow", "storage_pct"]:
    if col not in forecast_df.columns:
        continue
    for window in [3, 7, 14, 30]:
        forecast_df[f"{col}_rollmean_{window}"] = (
            forecast_df.groupby("lake_code")[col]
            .transform(lambda s: s.rolling(window, min_periods=2).mean())
        )
        forecast_df[f"{col}_rollstd_{window}"] = (
            forecast_df.groupby("lake_code")[col]
            .transform(lambda s: s.rolling(window, min_periods=2).std())
        )

for lag in [1, 3, 7, 14]:
    forecast_df[f"delta_level_{lag}d"] = forecast_df.groupby("lake_code")["water_level"].diff(lag)

forecast_df["lake_code_cat"] = forecast_df["lake_code"].astype("category").cat.codes

target_horizon = 7
target_col = f"target_t_plus_{target_horizon}"
forecast_df[target_col] = forecast_df.groupby("lake_code")["water_level"].shift(-target_horizon)

exclude_cols = {
    "timestamp", "lake_name", "province", "basin_name", "trend_text",
    "inflow_state", "water_level_state", "delta_state",
    "type", "Thời gian (UTC)", target_col
}

candidate_features = []
for col in forecast_df.columns:
    if col in exclude_cols:
        continue
    if col.startswith("target_t_plus_"):
        continue
    if forecast_df[col].dtype.kind in "biufc":
        candidate_features.append(col)

missing_ratio = forecast_df[candidate_features].isna().mean()
candidate_features = [c for c in candidate_features if missing_ratio[c] <= 0.35]
candidate_features = [c for c in candidate_features if forecast_df[c].nunique(dropna=True) > 1]

model_df = forecast_df[["timestamp", "lake_code", "lake_name", target_col] + candidate_features].copy()
model_df = model_df.dropna(subset=[target_col])

cutoff_date = model_df["timestamp"].quantile(0.8)
train_df = model_df[model_df["timestamp"] <= cutoff_date].copy()
test_df = model_df[model_df["timestamp"] > cutoff_date].copy()

fill_values = train_df[candidate_features].median(numeric_only=True)
X_train_full = train_df[candidate_features].fillna(fill_values)
X_test_full = test_df[candidate_features].fillna(fill_values)
y_train = train_df[target_col]
y_test = test_df[target_col]

mi = mutual_info_regression(X_train_full, y_train, random_state=42)
mi_df = pd.DataFrame({"feature": candidate_features, "mutual_info": mi}).sort_values("mutual_info", ascending=False)
selected_features = mi_df[mi_df["mutual_info"] > 0].head(25)["feature"].tolist()
if len(selected_features) < 10:
    selected_features = mi_df.head(min(15, len(mi_df)))["feature"].tolist()

corr_matrix = X_train_full[selected_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
selected_features = [c for c in selected_features if c not in to_drop]

X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

def evaluate_forecast(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred),
    }

results = []
importance_tables = []

baseline_col = f"water_level_lag_{target_horizon}" if f"water_level_lag_{target_horizon}" in test_df.columns else "water_level_lag_1"
baseline_pred = test_df[baseline_col].fillna(train_df["water_level"].median())
baseline_metrics = evaluate_forecast(y_test, baseline_pred)
results.append({"model": f"baseline_{baseline_col}", **baseline_metrics})

trained_models = {}

try:
    from lightgbm import LGBMRegressor
    lgbm = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    lgbm.fit(X_train, y_train)
    pred_lgbm = lgbm.predict(X_test)
    results.append({"model": "LightGBM", **evaluate_forecast(y_test, pred_lgbm)})
    importance_tables.append(
        pd.DataFrame({"feature": selected_features, "importance": lgbm.feature_importances_, "model": "LightGBM"})
        .sort_values("importance", ascending=False)
    )
    trained_models["LightGBM"] = lgbm
except Exception as e:
    print(f"Khong the train LightGBM: {e}")

try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(
        n_estimators=500,
        learning_rate=0.003,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        objective="reg:squarederror"
    )
    xgb.fit(X_train, y_train)
    pred_xgb = xgb.predict(X_test)
    results.append({"model": "XGBoost", **evaluate_forecast(y_test, pred_xgb)})
    importance_tables.append(
        pd.DataFrame({"feature": selected_features, "importance": xgb.feature_importances_, "model": "XGBoost"})
        .sort_values("importance", ascending=False)
    )
    trained_models["XGBoost"] = xgb
except Exception as e:
    print(f"Khong the train XGBoost: {e}")

metrics_df = pd.DataFrame(results).sort_values("RMSE")
metrics_df.to_csv(FORECAST_DIR / "forecast_model_metrics.csv", index=False, encoding="utf-8-sig")
mi_df.to_csv(FORECAST_DIR / "feature_selection_mutual_info.csv", index=False, encoding="utf-8-sig")

if importance_tables:
    feature_ranking = pd.concat(importance_tables, ignore_index=True)
    feature_ranking.to_csv(FORECAST_DIR / "feature_importance_ranking.csv", index=False, encoding="utf-8-sig")
    print(feature_ranking.sort_values(["model", "importance"], ascending=[True, False]).groupby("model").head(10))
else:
    feature_ranking = pd.DataFrame(columns=["feature", "importance", "model"])

selected_feature_table = pd.DataFrame({"selected_feature": selected_features})
selected_feature_table.to_csv(FORECAST_DIR / "selected_features.csv", index=False, encoding="utf-8-sig")

print(f"Ho duoc chon: {top_lake_name} | So ban ghi: {top_lake_n}")
print(f"Target forecast horizon: t+{target_horizon} ngay")
print(f"Train size: {len(train_df):,} | Test size: {len(test_df):,}")
print("Top feature theo mutual information:")
print(mi_df.head(15))
print("Selected features sau khi loai trung lap:")
print(selected_features)
print("Ket qua model:")
print(metrics_df)

/var/folders/61/d101nrlx4vv18pb88pttghhm0000gn/T/ipykernel_6025/3681801937.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  forecast_df[f"delta_level_{lag}d"] = forecast_df.groupby("lake_code")["water_level"].diff(lag)
/var/folders/61/d101nrlx4vv18pb88pttghhm0000gn/T/ipykernel_6025/3681801937.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  forecast_df["lake_code_cat"] = forecast_df["lake_code"].astype("category").cat.codes
/var/folders/61/d101nrlx4vv18pb88pttghhm0000gn/T/ipykernel_6025/3681801937.py:62: PerformanceWarni

Khong the train LightGBM: dlopen(/Users/cuongem/Desktop/Lear/Master/Quarter 4/Data-Mining/RiverTrackingAutomate/env/lib/python3.12/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <D44045CD-B874-3A27-9A61-F131D99AACE4> /Users/cuongem/Desktop/Lear/Master/Quarter 4/Data-Mining/RiverTrackingAutomate/env/lib/python3.12/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local